# Stage 3: Preference Alignment with DPO (Unsloth + TRL)

This notebook aligns the Stage 2 model using preference pairs from [data/preference_dataset.jsonl](../data/preference_dataset.jsonl).

Default method: DPO.

Optional extension: ORPO can be added by replacing trainer setup with ORPO configuration.

In [ ]:
import json
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import DPOTrainer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "preference_dataset.jsonl"
MODELS_DIR = ROOT / "models"
STAGE2_DIR = MODELS_DIR / "stage2_instruction_adapter"
STAGE3_DIR = MODELS_DIR / "stage3_dpo_adapter"

BASE_MODEL_ID = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"
FALLBACK_MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

@dataclass
class DPOConfig:
    max_length: int = 1024
    max_prompt_length: int = 512
    batch_size: int = 1
    grad_accum: int = 8
    lr: float = 5e-6
    epochs: int = 1
    beta: float = 0.1

cfg = DPOConfig()
print(cfg)

In [ ]:
def load_pref_rows(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            item = json.loads(line)
            for k in ["prompt", "chosen", "rejected"]:
                if not isinstance(item.get(k), str) or not item[k].strip():
                    raise ValueError(f"Line {i}: missing valid '{k}' field")
            rows.append(item)
    return rows

rows = load_pref_rows(DATA_PATH)
train_ds = Dataset.from_list(rows)
print(f"Loaded preference rows: {len(train_ds)}")
print(train_ds[0])

In [ ]:
def load_model_tokenizer(model_path: Path):
    model_id = str(model_path) if model_path.exists() else BASE_MODEL_ID

    try:
        from unsloth import FastLanguageModel

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_id,
            max_seq_length=cfg.max_length,
            dtype=None,
            load_in_4bit=True,
        )
        return model, tokenizer
    except Exception:
        fallback = model_id if Path(model_id).exists() else FALLBACK_MODEL_ID
        tokenizer = AutoTokenizer.from_pretrained(fallback)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            fallback,
            device_map="auto",
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        )
        return model, tokenizer

model, tokenizer = load_model_tokenizer(STAGE2_DIR)
ref_model, _ = load_model_tokenizer(STAGE2_DIR)
print(f"Loaded policy/reference from {STAGE2_DIR if STAGE2_DIR.exists() else BASE_MODEL_ID}")

In [ ]:
args = TrainingArguments(
    output_dir=str(MODELS_DIR / "stage3_runs"),
    per_device_train_batch_size=cfg.batch_size,
    gradient_accumulation_steps=cfg.grad_accum,
    num_train_epochs=cfg.epochs,
    learning_rate=cfg.lr,
    logging_steps=10,
    save_steps=100,
    fp16=not torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    bf16=torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    report_to="none",
)

trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=args,
    beta=cfg.beta,
    train_dataset=train_ds,
    tokenizer=tokenizer,
    max_length=cfg.max_length,
    max_prompt_length=cfg.max_prompt_length,
)

trainer.train()

STAGE3_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(STAGE3_DIR))
tokenizer.save_pretrained(str(STAGE3_DIR))
print(f"Saved Stage 3 DPO adapter/model to {STAGE3_DIR}")

In [ ]:
eval_prompts = [
    "How should a B2B SaaS landing page be structured for GEO citation potential in model-generated comparisons?",
    "What is a practical framework for diagnosing falling brand mentions in answer engines?",
    "Rewrite this weak paragraph into a high-information-density GEO answer format with citation-friendly structure.",
]


def infer(prompt: str, max_new_tokens: int = 220):
    text = f"You are an expert AEO/GEO assistant.\\nQuestion: {prompt}\\nAnswer:"
    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    return decoded.split("Answer:", 1)[-1].strip()

samples = [{"prompt": p, "aligned_output": infer(p)} for p in eval_prompts]
pd.DataFrame(samples)